In [2]:
# ============================================================================
# CELL 1: LOAD YOUR DATA
# ============================================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Set paths
DATA_DIR = Path("data/processed")

# Load your sequence data
print("Loading data...")
data = np.load(DATA_DIR / "sequences.npz")

X_train = data['X_train']  # (956, 986, 129)
X_val = data['X_val']      # (582, 986, 129)
X_test = data['X_test']    # (584, 986, 129)
y_val = data['y_val']
y_test = data['y_test']

print(f"✓ Data loaded successfully!")
print(f"  Train: {X_train.shape}")
print(f"  Validation: {X_val.shape}")
print(f"  Test: {X_test.shape}")
print(f"  Test anomaly rate: {y_test.mean():.1%}")

Loading data...
✓ Data loaded successfully!
  Train: (956, 986, 129)
  Validation: (582, 986, 129)
  Test: (584, 986, 129)
  Test anomaly rate: 64.7%


In [3]:
# ============================================================================
# CELL 2: UNDERSTAND YOUR DATA
# ============================================================================
print("="*60)
print("DATA ANALYSIS")
print("="*60)

print(f"\nData shape: {X_train.shape}")
print(f"  - Samples: {X_train.shape[0]}")
print(f"  - Timesteps: {X_train.shape[1]} (986 time points)")
print(f"  - Sensors: {X_train.shape[2]} (129 features)")

# Check for issues
print(f"\nData Quality:")
print(f"  - Missing values: {np.isnan(X_train).sum()}")
print(f"  - Infinite values: {np.isinf(X_train).sum()}")
print(f"  - Zeros: {(X_train == 0).sum()} ({(X_train == 0).sum() / X_train.size * 100:.2f}%)")

# Sensor statistics
sensor_means = X_train.mean(axis=(0, 1))
sensor_stds = X_train.std(axis=(0, 1))
sensor_mins = X_train.min(axis=(0, 1))
sensor_maxs = X_train.max(axis=(0, 1))

print(f"\nSensor Statistics:")
print(f"  - Mean range: [{sensor_means.min():.4f}, {sensor_means.max():.4f}]")
print(f"  - Std range: [{sensor_stds.min():.4f}, {sensor_stds.max():.4f}]")
print(f"  - Value range: [{sensor_mins.min():.4f}, {sensor_maxs.max():.4f}]")

# Identify constant sensors (no variation)
constant_sensors = np.where(sensor_stds < 1e-6)[0]
print(f"\nConstant sensors (no variation): {len(constant_sensors)}")
if len(constant_sensors) > 0:
    print(f"  First 10: {constant_sensors[:10]}")

DATA ANALYSIS

Data shape: (956, 986, 129)
  - Samples: 956
  - Timesteps: 986 (986 time points)
  - Sensors: 129 (129 features)

Data Quality:
  - Missing values: 0
  - Infinite values: 0
  - Zeros: 0 (0.00%)

Sensor Statistics:
  - Mean range: [-0.0000, 0.0001]
  - Std range: [0.9963, 1.0030]
  - Value range: [-28.1004, 51.7204]

Constant sensors (no variation): 0


In [4]:
# ============================================================================
# CELL 3: OPTIMIZED STATISTICAL FEATURES
# ============================================================================
import warnings
warnings.filterwarnings('ignore')

def extract_statistical_features_efficient(sequences):
    """
    Efficient statistical feature extraction using numpy operations.
    Much faster than looping through each sensor individually.
    """
    n_samples = sequences.shape[0]
    n_timesteps = sequences.shape[1]
    n_features = sequences.shape[2]
    
    # Pre-allocate feature array
    # 10 statistical features per sensor
    n_stats_per_sensor = 10
    n_total_features = n_stats_per_sensor * n_features
    features = np.zeros((n_samples, n_total_features))
    
    for i in range(n_samples):
        sample = sequences[i]  # (timesteps, features)
        
        # Compute all statistics at once using numpy
        stats_list = [
            np.mean(sample, axis=0),           # Mean
            np.std(sample, axis=0),            # Std
            np.median(sample, axis=0),         # Median
            np.min(sample, axis=0),            # Min
            np.max(sample, axis=0),            # Max
            np.ptp(sample, axis=0),            # Range
            np.percentile(sample, 25, axis=0), # Q1
            np.percentile(sample, 75, axis=0), # Q3
            np.percentile(sample, 90, axis=0), # P90
            np.percentile(sample, 95, axis=0), # P95
        ]
        
        # Concatenate all statistics
        features[i] = np.concatenate(stats_list)
    
    return features

print("\n" + "="*60)
print("EXTRACTING STATISTICAL FEATURES (Optimized)")
print("="*60)

X_train_stats = extract_statistical_features_efficient(X_train)
X_val_stats = extract_statistical_features_efficient(X_val)
X_test_stats = extract_statistical_features_efficient(X_test)

print(f"✓ Statistical features extracted!")
print(f"  Shape: {X_train_stats.shape}")
print(f"  (10 stats × 129 sensors = {10*129} features)")


EXTRACTING STATISTICAL FEATURES (Optimized)
✓ Statistical features extracted!
  Shape: (956, 1290)
  (10 stats × 129 sensors = 1290 features)


In [5]:
# ============================================================================
# CELL 4: EXTREME VALUE FEATURES (Captures anomalies)
# ============================================================================
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

def extract_extreme_features(sequences):
    """
    Focus on extreme values - anomalies often show up as outliers.
    """
    n_samples = sequences.shape[0]
    n_features = sequences.shape[2]
    
    features = np.zeros((n_samples, n_features * 6))
    
    for i in range(n_samples):
        if i % 200 == 0:
            print(f"  Processing sample {i}/{n_samples}")
        
        sample = sequences[i]
        
        # For each sensor, count extreme events
        extreme_stats = []
        for f in range(n_features):
            sensor_data = sample[:, f]
            
            # Count values beyond 2, 3, 4 standard deviations
            std = np.std(sensor_data)
            if std > 0:
                beyond_2sigma = np.sum(np.abs(sensor_data) > 2 * std) / len(sensor_data)
                beyond_3sigma = np.sum(np.abs(sensor_data) > 3 * std) / len(sensor_data)
                beyond_4sigma = np.sum(np.abs(sensor_data) > 4 * std) / len(sensor_data)
            else:
                beyond_2sigma = beyond_3sigma = beyond_4sigma = 0
            
            # Max absolute value
            max_abs = np.max(np.abs(sensor_data))
            
            # Skewness (asymmetry - anomalies often skew distribution)
            try:
                skew = stats.skew(sensor_data)
            except:
                skew = 0
            
            # Kurtosis (tail heaviness)
            try:
                kurt = stats.kurtosis(sensor_data)
            except:
                kurt = 0
            
            extreme_stats.extend([beyond_2sigma, beyond_3sigma, beyond_4sigma, max_abs, skew, kurt])
        
        features[i] = extreme_stats
    
    return features

print("\n" + "="*60)
print("EXTRACTING EXTREME VALUE FEATURES")
print("="*60)

X_train_extreme = extract_extreme_features(X_train)
X_val_extreme = extract_extreme_features(X_val)
X_test_extreme = extract_extreme_features(X_test)

print(f"\n✓ Extreme features extracted!")
print(f"  Shape: {X_train_extreme.shape}")
print(f"  (6 stats × 129 sensors = {6*129} features)")


EXTRACTING EXTREME VALUE FEATURES
  Processing sample 0/956
  Processing sample 200/956
  Processing sample 400/956
  Processing sample 600/956
  Processing sample 800/956
  Processing sample 0/582
  Processing sample 200/582
  Processing sample 400/582
  Processing sample 0/584
  Processing sample 200/584
  Processing sample 400/584

✓ Extreme features extracted!
  Shape: (956, 774)
  (6 stats × 129 sensors = 774 features)


In [6]:
# ============================================================================
# CELL 5: TEMPORAL CHANGE FEATURES
# ============================================================================

def extract_change_features(sequences):
    """
    Capture how quickly sensor values change over time.
    Anomalies often cause sudden changes.
    """
    n_samples = sequences.shape[0]
    n_features = sequences.shape[2]
    
    features = np.zeros((n_samples, n_features * 5))
    
    for i in range(n_samples):
        if i % 200 == 0:
            print(f"  Processing sample {i}/{n_samples}")
        
        sample = sequences[i]
        
        change_stats = []
        for f in range(n_features):
            sensor_data = sample[:, f]
            
            # Differences
            diff = np.diff(sensor_data)
            abs_diff = np.abs(diff)
            
            # Change statistics
            change_stats.extend([
                np.mean(abs_diff),           # Average change per step
                np.std(diff) if len(diff) > 0 else 0,  # Volatility of changes
                np.max(abs_diff) if len(abs_diff) > 0 else 0,  # Maximum single change
                np.percentile(abs_diff, 95) if len(abs_diff) > 0 else 0,  # 95th percentile
                np.sum(abs_diff > 2 * np.std(abs_diff)) / len(abs_diff) if len(abs_diff) > 0 and np.std(abs_diff) > 0 else 0,  # Sudden changes
            ])
        
        features[i] = change_stats
    
    return features

print("\n" + "="*60)
print("EXTRACTING TEMPORAL CHANGE FEATURES")
print("="*60)

X_train_change = extract_change_features(X_train)
X_val_change = extract_change_features(X_val)
X_test_change = extract_change_features(X_test)

print(f"\n✓ Change features extracted!")
print(f"  Shape: {X_train_change.shape}")
print(f"  (5 stats × 129 sensors = {5*129} features)")


EXTRACTING TEMPORAL CHANGE FEATURES
  Processing sample 0/956
  Processing sample 200/956
  Processing sample 400/956
  Processing sample 600/956
  Processing sample 800/956
  Processing sample 0/582
  Processing sample 200/582
  Processing sample 400/582
  Processing sample 0/584
  Processing sample 200/584
  Processing sample 400/584

✓ Change features extracted!
  Shape: (956, 645)
  (5 stats × 129 sensors = 645 features)


In [7]:
# ============================================================================
# CELL 6: COMBINE ALL FEATURES AND TRAIN
# ============================================================================
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, average_precision_score
import pandas as pd

print("\n" + "="*60)
print("COMBINING FEATURES")
print("="*60)

# Combine all feature sets
X_train_combined = np.hstack([
    X_train_stats,
    X_train_extreme,
    X_train_change,
])

X_val_combined = np.hstack([
    X_val_stats,
    X_val_extreme,
    X_val_change,
])

X_test_combined = np.hstack([
    X_test_stats,
    X_test_extreme,
    X_test_change,
])

print(f"Combined features shape: {X_train_combined.shape}")
print(f"  - Statistical: {X_train_stats.shape[1]} features")
print(f"  - Extreme: {X_train_extreme.shape[1]} features")
print(f"  - Change: {X_train_change.shape[1]} features")
print(f"  - Total: {X_train_combined.shape[1]} features")

# Combine train and validation for training
X_train_full = np.vstack([X_train_combined, X_val_combined])
y_train_full = np.hstack([np.zeros(len(X_train_combined)), y_val])

print(f"\nTraining set size: {len(X_train_full)}")
print(f"  Normal samples: {np.sum(y_train_full == 0)}")
print(f"  Anomaly samples: {np.sum(y_train_full == 1)}")
print(f"  Anomaly rate: {np.mean(y_train_full):.1%}")

# Standardize features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_full)
X_test_scaled = scaler.transform(X_test_combined)

# Train Random Forest
print("\n" + "="*60)
print("TRAINING RANDOM FOREST")
print("="*60)

rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=20,
    min_samples_split=20,
    min_samples_leaf=10,
    random_state=42,
    n_jobs=-1,
    class_weight='balanced'
)

rf.fit(X_train_scaled, y_train_full)

# Predict
y_pred_proba = rf.predict_proba(X_test_scaled)[:, 1]
y_pred = (y_pred_proba > 0.5).astype(int)

# Metrics
test_auc = roc_auc_score(y_test, y_pred_proba)
test_pr_auc = average_precision_score(y_test, y_pred_proba)

print(f"\n RANDOM FOREST RESULTS:")
print(f"  ROC-AUC: {test_auc:.4f}")
print(f"  PR-AUC:  {test_pr_auc:.4f}")

print(f"\n Classification Report (threshold=0.5):")
print(classification_report(y_test, y_pred, target_names=['Normal', 'Anomaly']))

# Confusion Matrix
tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
print(f"\n Confusion Matrix:")
print(f"  ┌─────────────────────┬──────────┬──────────┐")
print(f"  │                     │ Predicted│ Predicted│")
print(f"  │                     │  Normal  │  Anomaly │")
print(f"  ├─────────────────────┼──────────┼──────────┤")
print(f"  │ Actual Normal       │    {tn:3d}    │    {fp:3d}    │")
print(f"  ├─────────────────────┼──────────┼──────────┤")
print(f"  │ Actual Anomaly      │    {fn:3d}    │    {tp:3d}    │")
print(f"  └─────────────────────┴──────────┴──────────┘")

print(f"\n Key Metrics:")
print(f"  Precision: {tp/(tp+fp):.4f}")
print(f"  Recall:    {tp/(tp+fn):.4f}")
print(f"  F1-Score:  {2*tp/(2*tp+fp+fn):.4f}")
print(f"  Accuracy:  {(tp+tn)/(tp+tn+fp+fn):.4f}")


COMBINING FEATURES
Combined features shape: (956, 2709)
  - Statistical: 1290 features
  - Extreme: 774 features
  - Change: 645 features
  - Total: 2709 features

Training set size: 1538
  Normal samples: 1161
  Anomaly samples: 377
  Anomaly rate: 24.5%

TRAINING RANDOM FOREST

 RANDOM FOREST RESULTS:
  ROC-AUC: 0.9890
  PR-AUC:  0.9926

 Classification Report (threshold=0.5):
              precision    recall  f1-score   support

      Normal       0.99      0.92      0.95       206
     Anomaly       0.96      1.00      0.98       378

    accuracy                           0.97       584
   macro avg       0.98      0.96      0.97       584
weighted avg       0.97      0.97      0.97       584


 Confusion Matrix:
  ┌─────────────────────┬──────────┬──────────┐
  │                     │ Predicted│ Predicted│
  │                     │  Normal  │  Anomaly │
  ├─────────────────────┼──────────┼──────────┤
  │ Actual Normal       │    189    │     17    │
  ├─────────────────────┼───

In [8]:
# ============================================================================
# SAVE THE MODEL
# ============================================================================
import joblib
from pathlib import Path

MODEL_DIR = Path("models/best_model")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# Save model, scaler, and feature info
joblib.dump(rf, MODEL_DIR / "random_forest_best.pkl")
joblib.dump(scaler, MODEL_DIR / "scaler.pkl")

# Save feature information
import json
feature_info = {
    'model_type': 'Random Forest',
    'n_features': X_train_combined.shape[1],
    'n_estimators': 300,
    'max_depth': 20,
    'test_roc_auc': 0.9890,
    'test_pr_auc': 0.9926,
    'precision': 0.9569,
    'recall': 0.9974,
    'f1_score': 0.9767,
    'accuracy': 0.9692
}

with open(MODEL_DIR / "model_info.json", 'w') as f:
    json.dump(feature_info, f, indent=2)

print(f"✓ Best model saved to {MODEL_DIR}")

✓ Best model saved to models/best_model


In [9]:
# ============================================================================
# ANALYZE THE MISCLASSIFICATIONS
# ============================================================================
print("\n" + "="*60)
print("ANALYZING MISCLASSIFICATIONS")
print("="*60)

# Get prediction probabilities
y_pred_proba = rf.predict_proba(X_test_scaled)[:, 1]
y_pred = (y_pred_proba > 0.5).astype(int)

# Find false positives (normal predicted as anomaly)
fp_indices = np.where((y_test == 0) & (y_pred == 1))[0]
print(f"\nFalse Positives (Normal flagged as Anomaly): {len(fp_indices)}")
print(f"  Their anomaly probabilities: {y_pred_proba[fp_indices]}")

# Find false negatives (anomaly predicted as normal)
fn_indices = np.where((y_test == 1) & (y_pred == 0))[0]
print(f"\nFalse Negatives (Missed Anomalies): {len(fn_indices)}")
if len(fn_indices) > 0:
    print(f"  Their anomaly probabilities: {y_pred_proba[fn_indices]}")

# Check if these are borderline cases
print(f"\nProbability distribution of correct predictions:")
print(f"  Correct Normals (prob < 0.5): {np.mean(y_pred_proba[y_test==0][y_pred[y_test==0]==0]):.4f}")
print(f"  Correct Anomalies (prob > 0.5): {np.mean(y_pred_proba[y_test==1][y_pred[y_test==1]==1]):.4f}")


ANALYZING MISCLASSIFICATIONS

False Positives (Normal flagged as Anomaly): 17
  Their anomaly probabilities: [0.61278468 0.62566245 0.68249269 0.72265272 0.69864042 0.75434634
 0.82204183 0.81263987 0.63879043 0.66535167 0.57776266 0.61074574
 0.71235003 0.6989623  0.70026161 0.62841408 0.54842406]

False Negatives (Missed Anomalies): 1
  Their anomaly probabilities: [0.46836357]

Probability distribution of correct predictions:
  Correct Normals (prob < 0.5): 0.0760
  Correct Anomalies (prob > 0.5): 0.7993
